# 0. Setup: imports, drive \& paths

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import re
import time
import numpy as np
import pandas as pd

from tqdm import tqdm
from openai import OpenAI, RateLimitError
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
BASE_DIR = "/content/drive/MyDrive/NLP_Projects/03_DrugReview/01_R0/00_labelling"
print("Files in BASE_DIR:", os.listdir(BASE_DIR))

RAW_DATA_PATH = os.path.join(BASE_DIR, "output/drugscom_labelling_wcge50_ok_dedup_text_cond_ingCLEAN.csv")

Files in BASE_DIR: ['DrugName 4omini_Labeling.ipynb', 'LabellingAnalysis.ipynb', 'data', 'output', 'DrugReview_CleanComments.ipynb']


# 1. Load dataset

In [3]:
# Load the raw data from our config path into a generic DataFrame
df = pd.read_csv(RAW_DATA_PATH, index_col = 0)
print(f"Loaded raw data with {df.shape[0]} rows.")

Loaded raw data with 102061 rows.


In [4]:
df.head()

,drugName,ingredient_norm,ingredient_set_key,n_ingredients,condition,condition_norm,review,text_norm,rating,date,date_parsed,usefulCount,split,review_word_count,review_tokenlike_count,rx_status
uniqueID,,,,,,,,,,,,,,,,
124923,Allopurinol,allopurinol,allopurinol,1,Gout,gout,"""\r\n300 mg and colcris to reduce anything the...",""" 300 mg and colcris to reduce anything the mi...",3,18-Feb-17,2017-02-18,10,train,110,110,OK
127676,Phentermine,phentermine,phentermine,1,Weight Loss,weight loss,"""\r\nA previous poster Ryan Smith said &quot;I...",""" a previous poster ryan smith said ""i've seen...",10,16-Apr-17,2017-04-16,20,test,164,164,OK
81819,Liraglutide,liraglutide,liraglutide,1,Obesity,obesity,""" began with the minimum 0.6 mg dose of Saxend...",""" began with the minimum 0.6 mg dose of saxend...",10,18-Mar-17,2017-03-18,28,train,142,142,OK
170223,Quetiapine,quetiapine,quetiapine,1,Bipolar Disorde,bipolar disorde,""" Caused depression and negative, self defeati...",""" caused depression and negative, self defeati...",1,25-Aug-16,2016-08-25,2,train,76,76,OK
48454,Ethinyl estradiol / levonorgestrel,ethinyl estradiol / levonorgestrel,ethinyl estradiol / levonorgestrel,2,Birth Control,birth control,""" Ever since I started taking this birth contr...",""" ever since i started taking this birth contr...",1,30-Oct-17,2017-10-30,2,train,119,119,OK


In [5]:
# =========================
# Mental health dataset: count words (and simple token proxy) for `statement`
# =========================
import numpy as np
import pandas as pd

TEXT_COL = "text_norm"   # <- your text column
if TEXT_COL not in df.columns:
    raise KeyError(f"'{TEXT_COL}' not in df.columns. Available: {df.columns.tolist()[:30]} ...")

# Ensure string, handle NaN
s = df[TEXT_COL].fillna("").astype(str)

# Word count: split on whitespace
df["statement_word_count"] = s.str.split().str.len().astype(int)

# Optional: a slightly better proxy that counts "tokens" as word-like units (letters/numbers/apostrophes)
# (This avoids overcounting punctuation-only splits.)
df["statement_tokenlike_count"] = s.str.findall(r"[A-Za-z0-9']+").str.len().astype(int)

# Quick summary
wc = df["statement_word_count"]
tc = df["statement_tokenlike_count"]

print("Rows:", len(df))
print("Empty statements:", int((wc == 0).sum()))
print("\nWord count summary:")
print("  max:", int(wc.max()))
print("  p99:", int(wc.quantile(0.99)))
print("  p95:", int(wc.quantile(0.95)))
print("  median:", int(wc.median()))

print("\nToken-like count summary:")
print("  max:", int(tc.max()))
print("  p99:", int(tc.quantile(0.99)))
print("  p95:", int(tc.quantile(0.95)))
print("  median:", int(tc.median()))

# Inspect a few longest statements (sanity check)
topk = 5
long_idx = wc.sort_values(ascending=False).head(topk).index
print("\nTop longest examples (index, words, preview):")
for i in long_idx:
    preview = s.loc[i][:140].replace("\n", " ")
    print(f"- idx={i} words={int(wc.loc[i])} preview={preview!r}")

Rows: 102061
Empty statements: 0

Word count summary:
  max: 1894
  p99: 155
  p95: 147
  median: 104

Token-like count summary:
  max: 1875
  p99: 156
  p95: 148
  median: 105

Top longest examples (index, words, preview):
- idx=121004 words=1894 preview='"two and a half months ago i was prescribed venlafaxine to help prevent chronic migraines. it did help the migraines (reduced them by almost'
- idx=45000 words=1162 preview='"i don’t find a lot of positive stories about antidepressants, or i find stories where people are taking the antidepressant the wrong way. i'
- idx=216072 words=1107 preview='"my complicated experience with the insertion of the copper iud. it was "one of the most difficult & complicated iud insertions i\'ve had in '
- idx=79035 words=1103 preview='"okay ladies, i have decided to share my experience with plan b one step to hopefully help ease even just a few minds, because i know just h'
- idx=131240 words=881 preview='"i am a long time sufferer of frequent (chron

# 2. OpenAI client & shared label mapping

In [6]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # Set via Colab Secrets
client = OpenAI()  # uses OPENAI_API_KEY

# 3. Mini annotator (gpt-4o-mini) - batched

In [7]:
# ============================
# DrugReview AI labeling (10-score + 5-class + efficacy/safety + burden/cost)
# FULL DROP-IN JOB (REPLACEMENT)
# - resume/checkpoint
# - HARD LABEL ALWAYS = argmax(soft probs) (enforced in code)
# - Robust to model slips: aspect labels may come back as SENT5 (e.g., VERY_POSITIVE) -> coerced
# - Uses (review + ingredient_norm + condition_norm) as input context
# ============================

import os, json, time, random, hashlib
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from tqdm import tqdm

from openai import OpenAI
from openai import RateLimitError, APIError, APITimeoutError, BadRequestError

# ----------------------------
# Config
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/NLP_Projects/03_DrugReview/01_R0/00_labelling"
OUT_DIR  = os.path.join(BASE_DIR, "output")
os.makedirs(OUT_DIR, exist_ok=True)

ANNOTATION_MODEL = "gpt-4o-mini"

# If you already created df_final, set DF_NAME="df_final"; otherwise use "df"
DF_NAME = "df"  # change to "df_final" if that's your label-ready dataset in memory

ID_COL   = "uniqueID"          # stable id
TEXT_COL = "review"

# context columns (optional but recommended)
ING_COL  = "ingredient_norm"
COND_COL = "condition_norm"

# checkpoint + merged
OUT_CSV    = os.path.join(OUT_DIR, "drugscom_ai_labels_checkpoint.csv")
MERGED_CSV = os.path.join(OUT_DIR, "drugscom_plus_ai_labels.csv")

# batching / retries
BATCH_SIZE = 20
MAX_RETRIES_PER_BATCH = 3
SLEEP_BASE = 1.5
SLEEP_CAP  = 45.0
USE_JITTER = True
TEMPERATURE = 0.0
SEED = 7  # stored in output for audit; Responses API seed may not be supported in your env

SENT5_KEYS  = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]
ASPECT_KEYS = ["POSITIVE", "NEUTRAL", "NEGATIVE"]

# label argmax tie-break (per your prompt requirement)
TIEBREAK_ORDER = ["VERY_NEGATIVE","NEGATIVE","NEUTRAL","POSITIVE","VERY_POSITIVE"]
TIEBREAK_RANK = {k:i for i,k in enumerate(TIEBREAK_ORDER)}

def _now_utc_iso():
    return datetime.now(timezone.utc).isoformat()

def rating10_to_sent5(r: int) -> str:
    if r <= 2:  return "VERY_NEGATIVE"
    if r <= 4:  return "NEGATIVE"
    if r <= 6:  return "NEUTRAL"
    if r <= 8:  return "POSITIVE"
    return "VERY_POSITIVE"

def coerce_rating10(x) -> int:
    """
    Robust rating coercion:
    - if missing/unparseable -> 5
    - else clamp to [1, 10]
    """
    try:
        r = int(x)
    except Exception:
        return 5
    if r < 1:  return 1
    if r > 10: return 10
    return r

def _coerce_float(x):
    try:
        return float(x)
    except Exception:
        return None

def renorm_probs(probs: dict) -> dict:
    vals = []
    for k in SENT5_KEYS:
        v = _coerce_float(probs.get(k))
        if v is None or (isinstance(v, float) and np.isnan(v)):
            v = 0.0
        v = max(0.0, float(v))
        vals.append(v)
    s = float(sum(vals))
    if s <= 0:
        out = {k: 0.0 for k in SENT5_KEYS}
        out["NEUTRAL"] = 1.0
        return out
    return {k: vals[i] / s for i, k in enumerate(SENT5_KEYS)}

def force_label_argmax(probs: dict) -> str:
    # tie-break by specified order
    items = [(k, float(probs.get(k, 0.0))) for k in SENT5_KEYS]
    maxv = max(v for _, v in items)
    ties = [k for k, v in items if v == maxv]
    if len(ties) == 1:
        return ties[0]
    ties.sort(key=lambda k: TIEBREAK_RANK[k])
    return ties[0]

def _sleep(attempt: int):
    t = SLEEP_BASE * (2 ** attempt)
    if USE_JITTER:
        t *= (0.7 + 0.6 * random.random())
    time.sleep(min(t, SLEEP_CAP))

def _stable_text_hash(s: str) -> str:
    return hashlib.md5((s or "").encode("utf-8")).hexdigest()

# ----------------------------
# Ensure uniqueID is a real column (your screenshot shows it may be the index)
# ----------------------------
def ensure_id_column(df: pd.DataFrame, id_col: str) -> pd.DataFrame:
    df = df.copy()
    if id_col in df.columns:
        return df

    if getattr(df.index, "name", None) == id_col:
        return df.reset_index()

    # common colab case: index holds uniqueID but is unnamed; we do a safe heuristic
    # Only promote if index is all-digit and unique
    idx = df.index
    try:
        idx_str = idx.astype(str)
        if idx_str.str.match(r"^\d+$").all() and idx.is_unique:
            df = df.reset_index().rename(columns={"index": id_col})
            return df
    except Exception:
        pass

    raise KeyError(
        f"'{id_col}' not found in df.columns and could not safely infer from index. "
        f"Columns: {df.columns.tolist()[:20]}..."
    )

# ----------------------------
# Aspect coercion (FIX for your error: e.g., ai_efficacy=VERY_POSITIVE)
# ----------------------------
def coerce_aspect(x) -> str:
    """
    Coerce model slips:
      - VERY_POSITIVE -> POSITIVE
      - VERY_NEGATIVE -> NEGATIVE
      - POSITIVE/NEUTRAL/NEGATIVE pass through
      - otherwise -> NEUTRAL
    """
    if x is None:
        return "NEUTRAL"
    s = str(x).strip().upper()
    if s in ASPECT_KEYS:
        return s
    if s == "VERY_POSITIVE":
        return "POSITIVE"
    if s == "VERY_NEGATIVE":
        return "NEGATIVE"
    # sometimes model outputs SENT5 labels for aspects
    if s in SENT5_KEYS:
        if s in ("POSITIVE", "NEUTRAL", "NEGATIVE"):
            return s
        if s == "VERY_POSITIVE":
            return "POSITIVE"
        if s == "VERY_NEGATIVE":
            return "NEGATIVE"
    return "NEUTRAL"

# ----------------------------
# Prompt: MUST satisfy label==argmax(probs)
# (we still enforce in code regardless)
# ----------------------------
BATCH_SENTIMENT_INSTRUCTIONS = f"""
You are an expert annotator of patient drug reviews.
You are NOT a clinician and must NOT provide medical advice.
Your task is ONLY to infer sentiment and aspect signals expressed in each review text.

Each input item includes:
- review: string
- ingredient: string (may be empty)
- condition: string (may be empty)

Use ingredient and condition ONLY as context to interpret the review (do not hallucinate facts).

Return for each item:
1) ai_rating_10: integer 1..10 (1=worst, 10=best).
2) probs: probability distribution over 5 overall sentiment labels (sum to 1.0):
   {", ".join(SENT5_KEYS)}
3) label: one of {", ".join(SENT5_KEYS)}.
   IMPORTANT: label MUST equal argmax(probs).
   If tie, break ties by this order:
   {" > ".join(TIEBREAK_ORDER)}
4) ai_efficacy: POSITIVE / NEUTRAL / NEGATIVE
5) ai_safety:   POSITIVE / NEUTRAL / NEGATIVE
6) ai_burden:   POSITIVE / NEUTRAL / NEGATIVE
7) ai_cost:     POSITIVE / NEUTRAL / NEGATIVE

VALID VALUES FOR ASPECTS:
- ai_efficacy/ai_safety/ai_burden/ai_cost MUST be ONLY one of: POSITIVE, NEUTRAL, NEGATIVE
- Do NOT output VERY_POSITIVE or VERY_NEGATIVE for aspects.

Aspect definitions:
- efficacy POSITIVE: helped/worked/improved; NEGATIVE: did not work/worse; NEUTRAL: unclear/not mentioned/mixed.
- safety POSITIVE: no side effects/well tolerated; NEGATIVE: side effects/adverse events; NEUTRAL: unclear/not mentioned/mixed.
- burden POSITIVE: convenient/easy; NEGATIVE: inconvenient/hard to adhere/complex regimen; NEUTRAL: unclear/not mentioned/mixed.
- cost POSITIVE: affordable/covered; NEGATIVE: expensive/denied/high OOP; NEUTRAL: unclear/not mentioned/mixed.

Input format:
JSON array of objects:
{{"index": int, "review": str, "ingredient": str, "condition": str}}

Output format:
JSON array (same length, same order by index). Each element:
{{
  "index": int,
  "ai_rating_10": int,
  "probs": {{"VERY_NEGATIVE": float, "NEGATIVE": float, "NEUTRAL": float, "POSITIVE": float, "VERY_POSITIVE": float}},
  "label": "VERY_NEGATIVE|NEGATIVE|NEUTRAL|POSITIVE|VERY_POSITIVE",
  "ai_efficacy": "POSITIVE|NEUTRAL|NEGATIVE",
  "ai_safety": "POSITIVE|NEUTRAL|NEGATIVE",
  "ai_burden": "POSITIVE|NEUTRAL|NEGATIVE",
  "ai_cost": "POSITIVE|NEUTRAL|NEGATIVE"
}}

Rules:
- Output MUST be valid JSON only (no backticks, no extra text).
- Array length MUST match input length and preserve order by index.
- Probabilities must be non-negative and sum to 1.0 (within rounding).
- The "label" MUST equal argmax(probs) using the specified tie-break order.
""".strip()

# ----------------------------
# Validation/parsing (DO NOT fail batch for aspect slips; we coerce)
# ----------------------------
def validate_one(obj, expected_index: int):
    if not isinstance(obj, dict): return False, "not_dict"
    if obj.get("index") != expected_index: return False, f"bad_index:{obj.get('index')}"

    r = obj.get("ai_rating_10")
    # don't fail batch for rare model slips like 0/11; we'll coerce later
    try:
        _ = int(r)
    except Exception:
        return False, f"bad_rating_type:{r}"

    probs = obj.get("probs")
    if not isinstance(probs, dict): return False, "probs_not_dict"
    for k in SENT5_KEYS:
        if k not in probs: return False, f"missing_prob:{k}"
        v = _coerce_float(probs.get(k))
        if v is None: return False, f"bad_prob:{k}={probs.get(k)}"
        if float(v) < 0: return False, f"neg_prob:{k}={v}"

    lbl = obj.get("label")
    if lbl not in SENT5_KEYS: return False, f"bad_label:{lbl}"

    # aspects: only check field presence; actual values will be coerced
    for a in ["ai_efficacy","ai_safety","ai_burden","ai_cost"]:
        if a not in obj:
            return False, f"missing_{a}"

    return True, None

def validate_batch(arr, n_expected: int):
    if not isinstance(arr, list): return False, "output_not_list"
    if len(arr) != n_expected: return False, f"len_mismatch:{len(arr)}"
    for i, obj in enumerate(arr):
        ok, err = validate_one(obj, expected_index=i)
        if not ok: return False, f"item_{i}:{err}"
    return True, None

def parse_json_strict(s: str):
    s = (s or "").strip()
    return json.loads(s)

# ----------------------------
# OpenAI calls
# ----------------------------
client = OpenAI()

def call_model_for_batch(payload, model=ANNOTATION_MODEL):
    inp = json.dumps(payload, ensure_ascii=False)
    resp = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": BATCH_SENTIMENT_INSTRUCTIONS},
            {"role": "user", "content": inp},
        ],
        temperature=TEMPERATURE,
        # NOTE: your environment earlier threw on seed for Responses API; do not pass seed here.
    )
    return resp.output_text, resp

def annotate_batch_with_retry(payload, model=ANNOTATION_MODEL):
    """
    Robust batch annotate:
    - tolerates len mismatch (model returns < or > expected)
    - realigns outputs by 'index'
    - fills missing items with defaults (NEUTRAL, rating=5)
    - still enforces hard label = argmax(probs)
    """
    def _default_item(i: int):
        probs = {"VERY_NEGATIVE": 0.0, "NEGATIVE": 0.0, "NEUTRAL": 1.0, "POSITIVE": 0.0, "VERY_POSITIVE": 0.0}
        return {
            "index": int(i),
            "ai_rating_10": 5,
            "probs": probs,
            "label": force_label_argmax(probs),
            "ai_efficacy": "NEUTRAL",
            "ai_safety": "NEUTRAL",
            "ai_burden": "NEUTRAL",
            "ai_cost": "NEUTRAL",
        }

    def _coerce_one_item(o, fallback_idx: int):
        """
        Coerce a single raw model output row into our cleaned schema.
        If malformed, returns a default for fallback_idx.
        """
        try:
            if not isinstance(o, dict):
                return _default_item(fallback_idx)

            # index: accept int-like; else fallback
            idx_raw = o.get("index", fallback_idx)
            try:
                idx = int(idx_raw)
            except Exception:
                idx = int(fallback_idx)

            probs = renorm_probs(o.get("probs", {}))
            hard = force_label_argmax(probs)

            return {
                "index": int(idx),
                "ai_rating_10": coerce_rating10(o.get("ai_rating_10")),
                "probs": probs,
                "label": hard,
                "ai_efficacy": coerce_aspect(o.get("ai_efficacy")),
                "ai_safety":   coerce_aspect(o.get("ai_safety")),
                "ai_burden":   coerce_aspect(o.get("ai_burden")),
                "ai_cost":     coerce_aspect(o.get("ai_cost")),
            }
        except Exception:
            return _default_item(fallback_idx)

    last_err = None
    n_expected = len(payload)

    for attempt in range(MAX_RETRIES_PER_BATCH):
        try:
            out_text, resp = call_model_for_batch(payload, model=model)

            # parse JSON
            try:
                arr = parse_json_strict(out_text)
            except Exception as e:
                last_err = e
                _sleep(attempt)
                continue

            if not isinstance(arr, list):
                last_err = ValueError("output_not_list")
                _sleep(attempt)
                continue

            # Coerce each returned item, then map by index
            by_idx = {}
            for j, o in enumerate(arr):
                item = _coerce_one_item(o, fallback_idx=j)
                # keep first occurrence if duplicates
                by_idx.setdefault(int(item["index"]), item)

            # Build exact-length cleaned outputs (0..n_expected-1), fill missing with defaults
            cleaned = []
            for i in range(n_expected):
                item = by_idx.get(i)
                if item is None:
                    item = _default_item(i)
                else:
                    # enforce expected index to preserve strict alignment with payload
                    item = dict(item)
                    item["index"] = i
                cleaned.append(item)

            return cleaned, out_text, resp

        except (RateLimitError, APITimeoutError, APIError) as e:
            last_err = e
            _sleep(attempt)
        except (json.JSONDecodeError, BadRequestError, ValueError) as e:
            last_err = e
            _sleep(attempt)

    raise RuntimeError(f"Batch failed after retries. Last error: {repr(last_err)}")

# ----------------------------
# Resume: load checkpoint
# ----------------------------
def load_done_ids(out_csv: str, id_col: str):
    if not os.path.exists(out_csv):
        return set()
    prev = pd.read_csv(out_csv)
    if id_col not in prev.columns:
        return set()
    return set(prev[id_col].dropna().astype(str).tolist())

done_ids = load_done_ids(OUT_CSV, ID_COL)
print(f"Checkpoint exists: {os.path.exists(OUT_CSV)} | done rows: {len(done_ids)}")

# ----------------------------
# Get working df
# ----------------------------
if DF_NAME not in globals():
    raise NameError(f"{DF_NAME} not found in globals(). Set DF_NAME correctly (df or df_final).")

work = globals()[DF_NAME]
work = ensure_id_column(work, ID_COL)

# Basic required columns
need_cols = [ID_COL, TEXT_COL, ING_COL, COND_COL]
missing = [c for c in need_cols if c not in work.columns]
if missing:
    raise KeyError(f"work df missing columns: {missing}\nFound: {work.columns.tolist()[:30]} ...")

# Skip already labeled
work = work.copy()
work[ID_COL] = work[ID_COL].astype(str)
to_do = work.loc[~work[ID_COL].isin(done_ids)].copy().reset_index(drop=True)

print("Total rows:", len(work))
print("To label now:", len(to_do))

# ----------------------------
# Output formatting
# ----------------------------
def format_rows(batch_df, batch_outputs, raw_json_text, model_name):
    out_rows = []
    for i, (_, row) in enumerate(batch_df.iterrows()):
        o = batch_outputs[i]
        probs = o["probs"]
        out_rows.append({
            ID_COL: row[ID_COL],
            "text_hash": _stable_text_hash(str(row[TEXT_COL])),

            "ai_rating_10": coerce_rating10(o.get("ai_rating_10")),
            "ai_sent5_from_ai_rating10": rating10_to_sent5(int(o["ai_rating_10"])),

            # HARD LABEL = argmax(probs) (enforced)
            "ai_sent5_hard": o["label"],

            "ai_p_very_negative": float(probs["VERY_NEGATIVE"]),
            "ai_p_negative": float(probs["NEGATIVE"]),
            "ai_p_neutral": float(probs["NEUTRAL"]),
            "ai_p_positive": float(probs["POSITIVE"]),
            "ai_p_very_positive": float(probs["VERY_POSITIVE"]),

            "ai_efficacy": o["ai_efficacy"],
            "ai_safety": o["ai_safety"],
            "ai_burden": o["ai_burden"],
            "ai_cost": o["ai_cost"],

            "ai_model": model_name,
            "ai_seed": SEED,
            "ai_temp": TEMPERATURE,
            "ai_ts_utc": _now_utc_iso(),

            # batch-level raw output for audit/debug
            "ai_raw_json": raw_json_text,
        })
    return pd.DataFrame(out_rows)

# ----------------------------
# Main loop with incremental write
# ----------------------------
WRITE_HEADER = not os.path.exists(OUT_CSV)

pbar = tqdm(range(0, len(to_do), BATCH_SIZE), desc="Labeling batches")
for start in pbar:
    end = min(start + BATCH_SIZE, len(to_do))
    batch = to_do.iloc[start:end].copy()

    payload = []
    for i, (_, r) in enumerate(batch.iterrows()):
        payload.append({
            "index": i,
            "review": str(r[TEXT_COL] or ""),
            "ingredient": str(r.get(ING_COL, "") or ""),
            "condition": str(r.get(COND_COL, "") or ""),
        })

    outputs, raw_json, resp = annotate_batch_with_retry(payload, model=ANNOTATION_MODEL)
    out_df = format_rows(batch, outputs, raw_json, ANNOTATION_MODEL)

    out_df.to_csv(OUT_CSV, mode="a", header=WRITE_HEADER, index=False)
    WRITE_HEADER = False

print("✅ Done. Checkpoint saved to:", OUT_CSV)

# ----------------------------
# Merge labels back onto work df
# ----------------------------
labeled = pd.read_csv(OUT_CSV)
labeled[ID_COL] = labeled[ID_COL].astype(str)

merged = work.copy()
merged[ID_COL] = merged[ID_COL].astype(str)
merged = merged.merge(labeled, on=ID_COL, how="left")

print("Merged shape:", merged.shape)
print("Labeled rows:", int(merged["ai_rating_10"].notna().sum()), "/", len(merged))

merged.to_csv(MERGED_CSV, index=False)
print("✅ Saved merged dataset:", MERGED_CSV)

Checkpoint exists: True | done rows: 102061
Total rows: 102061
To label now: 0


Labeling batches: 0it [00:00, ?it/s]

✅ Done. Checkpoint saved to: /content/drive/MyDrive/NLP_Projects/03_DrugReview/01_R0/00_labelling/output/drugscom_ai_labels_checkpoint.csv


Merged shape: (102061, 37)
Labeled rows: 102061 / 102061
✅ Saved merged dataset: /content/drive/MyDrive/NLP_Projects/03_DrugReview/01_R0/00_labelling/output/drugscom_plus_ai_labels.csv
